# Notebook 02 — Xử lý và làm sạch dữ liệu

**Mục tiêu**: Làm sạch toàn bộ dữ liệu, lưu vào `data/processed/cleaned/`
- Xử lý missing values
- Xóa duplicates
- Chuẩn hóa kiểu dữ liệu (datetime, numeric)
- Xử lý outliers

**Tương ứng báo cáo**: Phần 2 — Chương 3 (Xử lý và chuẩn bị dữ liệu)

In [4]:
import sys
import importlib
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import src.data_loader as data_loader
import src.preprocessing as preprocessing
importlib.reload(data_loader)
importlib.reload(preprocessing)
load_all_raw = data_loader.load_all_raw
save_processed = data_loader.save_processed
summarize = preprocessing.summarize
drop_high_null_cols = preprocessing.drop_high_null_cols
fill_numeric_median = preprocessing.fill_numeric_median
fill_categorical_mode = preprocessing.fill_categorical_mode
remove_duplicates = preprocessing.remove_duplicates
parse_dates = preprocessing.parse_dates
remove_outliers_iqr = preprocessing.remove_outliers_iqr

%matplotlib inline

In [5]:
tables = load_all_raw()

## 1. Xử lý bảng Sales (target chính)

In [15]:
sales_raw = tables['sales'].copy()
print('Before cleaning:')
display(summarize(sales_raw, 'sales_raw'))

before_rows = len(sales_raw)
before_nulls = int(sales_raw.isna().sum().sum())

sales = sales_raw.rename(columns={
    'Date': 'date',
    'Revenue': 'revenue',
    'COGS': 'cogs',
})
sales = parse_dates(sales, ['date'])
sales = remove_duplicates(sales)
sales = sales.sort_values('date').reset_index(drop=True)

after_rows = len(sales)
after_nulls = int(sales.isna().sum().sum())
print(f'sales: rows {before_rows} -> {after_rows}, nulls {before_nulls} -> {after_nulls}')
print('After cleaning:')
display(summarize(sales, 'sales_clean'))
display(sales.head())

save_processed(sales, 'sales_clean.csv')

Before cleaning:

=== sales_raw | shape: (3833, 3) ===
           dtype  null_count  null_pct  unique      sample
Date      object           0       0.0    3833  2012-07-04
Revenue  float64           0       0.0    3833  5123547.94
COGS     float64           0       0.0    3833  3982991.19


,dtype,null_count,null_pct,unique,sample
Date,object,0,0.0,3833,2012-07-04
Revenue,float64,0,0.0,3833,5123547.94
COGS,float64,0,0.0,3833,3982991.19


sales: rows 3833 -> 3833, nulls 0 -> 0
After cleaning:

=== sales_clean | shape: (3833, 3) ===
                  dtype  null_count  null_pct  unique               sample
date     datetime64[ns]           0       0.0    3833  2012-07-04 00:00:00
revenue         float64           0       0.0    3833           5123547.94
cogs            float64           0       0.0    3833           3982991.19


,dtype,null_count,null_pct,unique,sample
date,datetime64[ns],0,0.0,3833,2012-07-04 00:00:00
revenue,float64,0,0.0,3833,5123547.94
cogs,float64,0,0.0,3833,3982991.19


,date,revenue,cogs
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\sales_clean.csv


## 2. Xử lý các bảng Master

In [18]:
master_tables = ['products', 'customers', 'promotions', 'geography']

for name in master_tables:
    df = tables[name].copy()
    before_rows = len(df)
    before_nulls = int(df.isna().sum().sum())

    print(f'\nBefore cleaning: {name}')
    display(summarize(df, f'{name}_raw'))

    df = remove_duplicates(df)

    date_cols = [col for col in df.columns if 'date' in col.lower()]
    if date_cols:
        df = parse_dates(df, date_cols)

    if name != 'promotions': # nghiệp vụ
        df = fill_numeric_median(df)
        df = fill_categorical_mode(df)

    after_rows = len(df)
    after_nulls = int(df.isna().sum().sum())
    print(f'{name}: rows {before_rows} -> {after_rows}, nulls {before_nulls} -> {after_nulls}')
    print(f'After cleaning: {name}')
    display(summarize(df, f'{name}_clean'))

    save_processed(df, f'{name}_clean.csv')
    print(f'{name}: saved')


Before cleaning: products

=== products_raw | shape: (2412, 8) ===
                dtype  null_count  null_pct  unique            sample
product_id      int64           0       0.0    2412               536
product_name   object           0       0.0    2172  SaigonFlex UC-01
category       object           0       0.0       4        Streetwear
segment        object           0       0.0       8          Everyday
size           object           0       0.0       4                 S
color          object           0       0.0      10             green
price         float64           0       0.0    1990          11059.65
cogs          float64           0       0.0    2381       9704.842875


,dtype,null_count,null_pct,unique,sample
product_id,int64,0,0.0,2412,536
product_name,object,0,0.0,2172,SaigonFlex UC-01
category,object,0,0.0,4,Streetwear
segment,object,0,0.0,8,Everyday
size,object,0,0.0,4,S
color,object,0,0.0,10,green
price,float64,0,0.0,1990,11059.65
cogs,float64,0,0.0,2381,9704.842875


products: rows 2412 -> 2412, nulls 0 -> 0
After cleaning: products

=== products_clean | shape: (2412, 8) ===
                dtype  null_count  null_pct  unique            sample
product_id      int64           0       0.0    2412               536
product_name   object           0       0.0    2172  SaigonFlex UC-01
category       object           0       0.0       4        Streetwear
segment        object           0       0.0       8          Everyday
size           object           0       0.0       4                 S
color          object           0       0.0      10             green
price         float64           0       0.0    1990          11059.65
cogs          float64           0       0.0    2381       9704.842875


,dtype,null_count,null_pct,unique,sample
product_id,int64,0,0.0,2412,536
product_name,object,0,0.0,2172,SaigonFlex UC-01
category,object,0,0.0,4,Streetwear
segment,object,0,0.0,8,Everyday
size,object,0,0.0,4,S
color,object,0,0.0,10,green
price,float64,0,0.0,1990,11059.65
cogs,float64,0,0.0,2381,9704.842875


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\products_clean.csv
products: saved

Before cleaning: customers

=== customers_raw | shape: (121930, 7) ===
                      dtype  null_count  null_pct  unique        sample
customer_id           int64           0       0.0  121930             1
zip                   int64           0       0.0   31491         15201
city                 object           0       0.0      42     Hai Phong
signup_date          object           0       0.0    3941    2021-12-30
gender               object           0       0.0       3        Female
age_group            object           0       0.0       5         35-44
acquisition_channel  object           0       0.0       6  social_media


...

Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\promotions_clean.csv
promotions: saved

Before cleaning: geography

=== geography_raw | shape: (39948, 4) ===
           dtype  null_count  null_pct  unique        sample
zip        int64           0       0.0   39948         15201
city      object           0       0.0      42     Hai Phong
region    object           0       0.0       3          East
district  object           0       0.0      39  District #13


,dtype,null_count,null_pct,unique,sample
zip,int64,0,0.0,39948,15201
city,object,0,0.0,42,Hai Phong
region,object,0,0.0,3,East
district,object,0,0.0,39,District #13


geography: rows 39948 -> 39948, nulls 0 -> 0
After cleaning: geography

=== geography_clean | shape: (39948, 4) ===
           dtype  null_count  null_pct  unique        sample
zip        int64           0       0.0   39948         15201
city      object           0       0.0      42     Hai Phong
region    object           0       0.0       3          East
district  object           0       0.0      39  District #13


,dtype,null_count,null_pct,unique,sample
zip,int64,0,0.0,39948,15201
city,object,0,0.0,42,Hai Phong
region,object,0,0.0,3,East
district,object,0,0.0,39,District #13


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\geography_clean.csv
geography: saved


## 3. Xử lý các bảng Transaction

In [19]:
transaction_tables = ['orders', 'order_items', 'payments', 'shipments', 'returns', 'reviews']
for name in transaction_tables:
    df = tables[name].copy()

    before_rows = len(df)
    before_nulls = int(df.isna().sum().sum())

    print((f"\nBefore cleaning: {name}"))
    display(summarize(df, f"{name}_raw"))

    df = remove_duplicates(df)

    date_cols = [col for col in df.columns if "date" in col.lower()]
    if date_cols:
        df = parse_dates(df, date_cols)

    if name == 'order_items':
        # Nghiệp vụ không fill cho order_items; giữ nguyên null để xử lý ở bước sau nếu cần
        pass

    if name != 'order_items': # nghiệp vụ
        df = fill_numeric_median(df)
        df = fill_categorical_mode(df)

    after_rows = len(df)
    after_nulls = int(df.isna().sum().sum())
    print(f"{name}: rows {before_rows} -> {after_rows}, nulls{before_nulls} -> {after_nulls}")
    display(f"After cleaning: {name}")
    display(summarize(df, f'{name}_clean'))

    save_processed(df, f'{name}_clean.csv')
    print(f'{name}: done')


Before cleaning: orders

=== orders_raw | shape: (646945, 8) ===
                 dtype  null_count  null_pct  unique       sample
order_id         int64           0       0.0  646945            1
order_date      object           0       0.0    3833   2012-07-04
customer_id      int64           0       0.0   90246        58578
zip              int64           0       0.0   29932         1109
order_status    object           0       0.0       6    delivered
payment_method  object           0       0.0       5  credit_card
device_type     object           0       0.0       3      desktop
order_source    object           0       0.0       6  paid_search


,dtype,null_count,null_pct,unique,sample
order_id,int64,0,0.0,646945,1
order_date,object,0,0.0,3833,2012-07-04
customer_id,int64,0,0.0,90246,58578
zip,int64,0,0.0,29932,1109
order_status,object,0,0.0,6,delivered
payment_method,object,0,0.0,5,credit_card
device_type,object,0,0.0,3,desktop
order_source,object,0,0.0,6,paid_search


orders: rows 646945 -> 646945, nulls0 -> 0


'After cleaning: orders'


=== orders_clean | shape: (646945, 8) ===
                         dtype  null_count  null_pct  unique               sample
order_id                 int64           0       0.0  646945                    1
order_date      datetime64[ns]           0       0.0    3833  2012-07-04 00:00:00
customer_id              int64           0       0.0   90246                58578
zip                      int64           0       0.0   29932                 1109
order_status            object           0       0.0       6            delivered
payment_method          object           0       0.0       5          credit_card
device_type             object           0       0.0       3              desktop
order_source            object           0       0.0       6          paid_search


...

reviews: rows 113551 -> 113551, nulls0 -> 0


'After cleaning: reviews'


=== reviews_clean | shape: (113551, 7) ===
                       dtype  null_count  null_pct  unique               sample
review_id             object           0       0.0  113551          REV-0000001
order_id               int64           0       0.0  111369                    1
product_id             int64           0       0.0    1412                 2400
customer_id            int64           0       0.0   48676                58578
review_date   datetime64[ns]           0       0.0    3825  2012-07-24 00:00:00
rating                 int64           0       0.0       5                    5
review_title          object           0       0.0      18     Highly recommend


,dtype,null_count,null_pct,unique,sample
review_id,object,0,0.0,113551,REV-0000001
order_id,int64,0,0.0,111369,1
product_id,int64,0,0.0,1412,2400
customer_id,int64,0,0.0,48676,58578
review_date,datetime64[ns],0,0.0,3825,2012-07-24 00:00:00
rating,int64,0,0.0,5,5
review_title,object,0,0.0,18,Highly recommend


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\reviews_clean.csv
reviews: done


## 4. Xử lý Operational (inventory, web_traffic)

In [20]:
# TODO: Xử lý inventory, web_traffic
for name in ['inventory', 'web_traffic']:
    df = tables[name].copy()
    df = remove_duplicates(df)

    before_rows = len(df)
    before_nulls = int(df.isna().sum().sum())

    print(f"\nBefore cleaning: {name}")
    display(summarize(df, f"{name}_raw"))

    df = remove_duplicates(df)

    date_cols = [col for col in df.columns if 'date' in col.lower()]
    if date_cols: 
        df = parse_dates(df, date_cols)
    
    df = fill_numeric_median(df)
    df = fill_categorical_mode(df)

    after_rows = len(df)
    after_nulls = int(df.isna().sum().sum())
    print(f"{name} : rows {before_rows} -> {after_rows}, nulls {before_nulls} -> {after_nulls}  ")
    print(f"After clearning: {name}")
    display(summarize(df, f'{name}_clean'))

    save_processed(df, f'{name}_clean.csv')
    print(f'{name}: done')


Before cleaning: inventory

=== inventory_raw | shape: (60247, 17) ===
                     dtype  null_count  null_pct  unique            sample
snapshot_date       object           0       0.0     126        2022-10-31
product_id           int64           0       0.0    1624                 1
stock_on_hand        int64           0       0.0    1895                 3
units_received       int64           0       0.0     360                 1
units_sold           int64           0       0.0     303                 1
stockout_days        int64           0       0.0      29                 2
days_of_supply     float64           0       0.0    9289              90.0
fill_rate          float64           0       0.0      29            0.9333
stockout_flag        int64           0       0.0       2                 1
overstock_flag       int64           0       0.0       2                 0
reorder_flag         int64           0       0.0       1                 0
sell_through_rate  float64  

,dtype,null_count,null_pct,unique,sample
snapshot_date,object,0,0.0,126,2022-10-31
product_id,int64,0,0.0,1624,1
stock_on_hand,int64,0,0.0,1895,3
units_received,int64,0,0.0,360,1
units_sold,int64,0,0.0,303,1
stockout_days,int64,0,0.0,29,2
days_of_supply,float64,0,0.0,9289,90.0
fill_rate,float64,0,0.0,29,0.9333
stockout_flag,int64,0,0.0,2,1
overstock_flag,int64,0,0.0,2,0


inventory : rows 60247 -> 60247, nulls 0 -> 0  
After clearning: inventory

=== inventory_clean | shape: (60247, 17) ===
                            dtype  null_count  null_pct  unique               sample
snapshot_date      datetime64[ns]           0       0.0     126  2022-10-31 00:00:00
product_id                  int64           0       0.0    1624                    1
stock_on_hand               int64           0       0.0    1895                    3
units_received              int64           0       0.0     360                    1
units_sold                  int64           0       0.0     303                    1
stockout_days               int64           0       0.0      29                    2
days_of_supply            float64           0       0.0    9289                 90.0
fill_rate                 float64           0       0.0      29               0.9333
stockout_flag               int64           0       0.0       2                    1
overstock_flag              i

,dtype,null_count,null_pct,unique,sample
snapshot_date,datetime64[ns],0,0.0,126,2022-10-31 00:00:00
product_id,int64,0,0.0,1624,1
stock_on_hand,int64,0,0.0,1895,3
units_received,int64,0,0.0,360,1
units_sold,int64,0,0.0,303,1
stockout_days,int64,0,0.0,29,2
days_of_supply,float64,0,0.0,9289,90.0
fill_rate,float64,0,0.0,29,0.9333
stockout_flag,int64,0,0.0,2,1
overstock_flag,int64,0,0.0,2,0


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\inventory_clean.csv
inventory: done

Before cleaning: web_traffic

=== web_traffic_raw | shape: (3652, 7) ===
                            dtype  null_count  null_pct  unique          sample
date                       object           0       0.0    3652      2013-01-01
sessions                    int64           0       0.0    3447            9760
unique_visitors             int64           0       0.0    3382            7253
page_views                  int64           0       0.0    3620           39093
bounce_rate               float64           0       0.0     261         0.00514
avg_session_duration_sec  float64           0       0.0    1771           102.9
traffic_source             object           0       0.0       6  organic_search


,dtype,null_count,null_pct,unique,sample
date,object,0,0.0,3652,2013-01-01
sessions,int64,0,0.0,3447,9760
unique_visitors,int64,0,0.0,3382,7253
page_views,int64,0,0.0,3620,39093
bounce_rate,float64,0,0.0,261,0.00514
avg_session_duration_sec,float64,0,0.0,1771,102.9
traffic_source,object,0,0.0,6,organic_search


web_traffic : rows 3652 -> 3652, nulls 0 -> 0  
After clearning: web_traffic

=== web_traffic_clean | shape: (3652, 7) ===
                                   dtype  null_count  null_pct  unique               sample
date                      datetime64[ns]           0       0.0    3652  2013-01-01 00:00:00
sessions                           int64           0       0.0    3447                 9760
unique_visitors                    int64           0       0.0    3382                 7253
page_views                         int64           0       0.0    3620                39093
bounce_rate                      float64           0       0.0     261              0.00514
avg_session_duration_sec         float64           0       0.0    1771                102.9
traffic_source                    object           0       0.0       6       organic_search


,dtype,null_count,null_pct,unique,sample
date,datetime64[ns],0,0.0,3652,2013-01-01 00:00:00
sessions,int64,0,0.0,3447,9760
unique_visitors,int64,0,0.0,3382,7253
page_views,int64,0,0.0,3620,39093
bounce_rate,float64,0,0.0,261,0.00514
avg_session_duration_sec,float64,0,0.0,1771,102.9
traffic_source,object,0,0.0,6,organic_search


Saved: d:\Code\DS_Code\revenue_forecast_ecommerce\notebooks\..\data\processed\cleaned\web_traffic_clean.csv
web_traffic: done


## 5. Tóm tắt kết quả làm sạch

| Bảng | Rows gốc | Rows sau clean | Null xử lý | Duplicate xóa |
|------|----------|----------------|------------|---------------|
| sales | 3,833 | 3,833 | 0 | 0 |
| products | 2,412 | 2,412 | 0 | 0 |
| customers | 121,930 | 121,930 | 0 | 0 |
| promotions | 50 | 50 | 0 | 0 |
| geography | 39,948 | 39,948 | 0 | 0 |
| orders | 646,945 | 646,945 | 0 | 0 |
| order_items | 714,669 | 714,669 | 0 | 0 |
| payments | 646,945 | 646,945 | 0 | 0 |
| shipments | 566,067 | 566,067 | 0 | 0 |
| returns | 39,939 | 39,939 | 0 | 0 |
| reviews | 113,551 | 113,551 | 0 | 0 |
| inventory | 60,247 | 60,247 | 0 | 0 |
| web_traffic | 3,652 | 3,652 | 0 | 0 |

Lưu ý: `promotions` và `order_items` giữ null theo nghiệp vụ nên không fill giá trị thiếu ở các cột đó.